# Notebook 06 — Optical Flow, Point Tracking & Scene Flow

**Vision & 3D Mapping Workshop** | Block 3: Multi-View Geometry

---

## Why This Matters

Motion is one of the most powerful cues available to a visual system. From a pair of images
taken a fraction of a second apart, we can infer how objects move, how the camera moves, and
even recover 3D scene structure. **Optical flow** — the apparent 2D motion field — is the
bridge between image measurements and 3D understanding.

### What You'll Learn

1. **Brightness constancy** — the foundational assumption and its limitations
2. **Lucas-Kanade** — sparse flow via least squares (from scratch)
3. **Pyramidal LK** — coarse-to-fine for large displacements
4. **Farneback** — dense flow via polynomial expansion
5. **RAFT** — the modern deep-learning flow architecture
6. **Point tracking** — long-range tracking with TAPIR and CoTracker
7. **Scene flow** — extending 2D flow to 3D motion vectors
8. **Exercises** — hands-on practice

### Prerequisites
- Notebook 01 (image gradients, convolution)
- Notebook 03 (camera intrinsics, projection)
- Linear algebra (least squares, eigenvalues)

### References
- Lucas & Kanade, "An Iterative Image Registration Technique" (1981)
- Farnebäck, "Two-Frame Motion Estimation Based on Polynomial Expansion" (2003)
- Teed & Deng, "RAFT: Recurrent All-Pairs Field Transforms for Optical Flow" (ECCV 2020)
- Doersch et al., "TAPIR: Tracking Any Point with per-frame Initialization and temporal Refinement" (ICCV 2023)
- Karaev et al., "CoTracker: It is Better to Track Together" (ECCV 2024)
- Vedula et al., "Three-dimensional scene flow" (ICCV 1999)

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import hsv_to_rgb
from mpl_toolkits.mplot3d import Axes3D
import cv2
from scipy.ndimage import gaussian_filter

%matplotlib inline
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["image.cmap"] = "gray"
np.set_printoptions(precision=4, suppress=True)

---
## 1. Brightness Constancy Assumption

### 1.1 The Fundamental Assumption

Optical flow estimation rests on a single, powerful assumption: **the brightness of a point
does not change as it moves across the image**. If a pixel at $(x, y)$ in frame $t$ moves
to $(x + \delta x, y + \delta y)$ in frame $t + \delta t$, then:

$$
I(x, y, t) = I(x + \delta x, y + \delta y, t + \delta t)
$$

This assumes no lighting changes, no specular reflections, no occlusions. It's approximate
in practice, but remarkably useful.

### 1.2 Deriving the Optical Flow Equation

Apply a **first-order Taylor expansion** to the right-hand side:

$$
I(x + \delta x, y + \delta y, t + \delta t) \approx I(x, y, t)
+ I_x \, \delta x + I_y \, \delta y + I_t \, \delta t
$$

where $I_x = \frac{\partial I}{\partial x}$, $I_y = \frac{\partial I}{\partial y}$,
$I_t = \frac{\partial I}{\partial t}$ are the spatial and temporal partial derivatives.

Substituting into the brightness constancy equation:

$$
I(x, y, t) = I(x, y, t) + I_x \, \delta x + I_y \, \delta y + I_t \, \delta t
$$

$$
\Rightarrow \quad I_x \, \delta x + I_y \, \delta y + I_t \, \delta t = 0
$$

**Divide by $\delta t$** and define the velocity components $u = \frac{dx}{dt}$,
$v = \frac{dy}{dt}$:

$$
\boxed{I_x \, u + I_y \, v + I_t = 0}
$$

This is the **optical flow constraint equation**. In compact vector form:

$$
\nabla I \cdot \mathbf{v} + I_t = 0
$$

where $\nabla I = (I_x, I_y)^\top$ is the spatial gradient and
$\mathbf{v} = (u, v)^\top$ is the flow vector.

> **Validity.** The Taylor expansion neglects terms of order $|u|^2, |v|^2$, so the
> linearization is accurate only for **small displacements** ($|u|, |v| \ll 1$ pixel).
> When objects move faster, we need a **coarse-to-fine pyramid** (Section 3) to reduce
> large motions to small ones at each scale level.

#### 1.2a When Brightness Constancy Fails

The BCA is violated in many real scenarios. Understanding *when* it fails is critical
for building robust autonomous systems:

| Violation | Cause | Mitigation |
|-----------|-------|------------|
| **Illumination change** | Sun moves, headlights, shadows | Gradient constancy: $\nabla I_x u + \nabla I_y v + \nabla I_t = 0$ (Brox et al., 2004) |
| **Specular reflection** | Shiny surfaces, wet roads | Use diffuse component only; learned features are more robust |
| **Occlusion** | Object boundaries | Forward-backward consistency check; detect and exclude occluded pixels |
| **Transparency** | Glass, water | Multiple motion models (layer decomposition) |
| **Non-Lambertian surfaces** | Metal, mirrors | Appearance-invariant descriptors (Census transform, RAFT) |

The **gradient constancy assumption** (Brox et al., ECCV 2004) is an important
generalization: instead of assuming pixel *values* are preserved, assume spatial
*gradients* are preserved. This is invariant to additive illumination changes
($I \to I + c$) and handles shadows/exposure changes much better. Modern learned
methods like RAFT implicitly learn which constancy assumption to apply via the
correlation volume.

### 1.3 The Aperture Problem

We have **one equation** but **two unknowns** ($u$ and $v$). This is fundamentally
under-determined — a single pixel cannot tell us both components of motion.

The constraint $I_x u + I_y v = -I_t$ defines a **line** in $(u, v)$ space, not a point.
Every velocity on this line is consistent with the observed brightness change.

The only component we *can* recover is the **normal flow** — the component of motion
along the image gradient direction:

$$
v_n = -\frac{I_t}{|\nabla I|}
$$

**Geometric interpretation**: Imagine looking at a moving edge through a small aperture
(like a keyhole). You can see motion *perpendicular* to the edge, but not *along* it.
A horizontal edge moving diagonally appears to move purely vertically.

To resolve the ambiguity, we need **additional constraints**:

| Method | Constraint | Equation | Reference |
|--------|-----------|----------|-----------|
| **Lucas-Kanade** | Constant flow in local window | Overdetermined linear system (Section 2) | Lucas & Kanade, 1981 |
| **Horn-Schunck** | Smooth flow globally | Variational: $E = \int (I_x u + I_y v + I_t)^2 + \alpha^2(\|\nabla u\|^2 + \|\nabla v\|^2)\,dA$ | Horn & Schunck, 1981 |
| **RAFT** | Learned correspondence prior | 4D correlation volume + iterative GRU refinement (Section 5) | Teed & Deng, 2020 |

The Horn-Schunck energy functional is one of the first applications of **variational
methods** in computer vision. Its Euler-Lagrange equations yield a pair of coupled PDEs
that are solved iteratively — this global smoothness assumption is the complement to
LK's local assumption. Modern methods (Brox et al., ECCV 2004) replace the quadratic
smoothness penalty with robust norms $\psi(|\nabla u|^2)$ to preserve flow
discontinuities at object boundaries.

In [ ]:
np.random.seed(42)

def make_translating_blobs(H=200, W=300, n_blobs=8, dx=3.0, dy=2.0):
    """Generate two frames of translating Gaussian blobs."""
    img1 = np.zeros((H, W), dtype=np.float64)
    img2 = np.zeros((H, W), dtype=np.float64)
    yy, xx = np.mgrid[0:H, 0:W].astype(np.float64)

    centers = np.column_stack([
        np.random.randint(30, W - 30, n_blobs),
        np.random.randint(30, H - 30, n_blobs)
    ]).astype(np.float64)
    sigmas = np.random.uniform(10, 25, n_blobs)
    amplitudes = np.random.uniform(0.5, 1.0, n_blobs)

    for (cx, cy), sigma, amp in zip(centers, sigmas, amplitudes):
        img1 += amp * np.exp(-((xx - cx)**2 + (yy - cy)**2) / (2 * sigma**2))
        img2 += amp * np.exp(-((xx - cx - dx)**2 + (yy - cy - dy)**2) / (2 * sigma**2))

    img1 = np.clip(img1, 0, 1)
    img2 = np.clip(img2, 0, 1)
    return img1, img2, dx, dy

img1, img2, true_dx, true_dy = make_translating_blobs()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(img1, cmap="gray")
axes[0].set_title("Frame 1")
axes[1].imshow(img2, cmap="gray")
axes[1].set_title("Frame 2")
axes[2].imshow(np.abs(img2 - img1), cmap="hot")
axes[2].set_title("|Frame 2 − Frame 1|")
for ax in axes:
    ax.axis("off")
plt.suptitle(f"Translating blobs: ground-truth flow = ({true_dx}, {true_dy}) px/frame",
             fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
def compute_gradients(img1, img2):
    """Compute spatial gradients (Sobel) and temporal gradient."""
    Ix = cv2.Sobel(img1, cv2.CV_64F, 1, 0, ksize=5) / 128.0
    Iy = cv2.Sobel(img1, cv2.CV_64F, 0, 1, ksize=5) / 128.0
    It = img2.astype(np.float64) - img1.astype(np.float64)
    return Ix, Iy, It

Ix, Iy, It = compute_gradients(img1, img2)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, data, title in [
    (axes[0], Ix, r"$I_x$ (horizontal gradient)"),
    (axes[1], Iy, r"$I_y$ (vertical gradient)"),
    (axes[2], It, r"$I_t$ (temporal gradient)"),
]:
    im = ax.imshow(data, cmap="RdBu_r")
    ax.set_title(title, fontsize=12)
    ax.axis("off")
    plt.colorbar(im, ax=ax, shrink=0.8)
plt.suptitle("Image Gradients for Optical Flow", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Constraint lines for several pixels ---
ax = axes[0]
sample_points = [(100, 80), (150, 100), (200, 60), (130, 150)]
u_range = np.linspace(-8, 12, 300)
colors = plt.cm.tab10(np.linspace(0, 1, len(sample_points)))

for (px, py), color in zip(sample_points, colors):
    ix_val = Ix[py, px]
    iy_val = Iy[py, px]
    it_val = It[py, px]
    if abs(iy_val) > 1e-8:
        v_line = -(ix_val * u_range + it_val) / iy_val
        ax.plot(u_range, v_line, color=color, linewidth=2,
                label=f"pixel ({px},{py})")
ax.axhline(0, color="gray", linewidth=0.5)
ax.axvline(0, color="gray", linewidth=0.5)
ax.plot(true_dx, true_dy, "k*", markersize=15, label="Ground truth", zorder=5)
ax.set_xlabel("u (px/frame)")
ax.set_ylabel("v (px/frame)")
ax.set_title("Optical Flow Constraint Lines in (u, v) Space")
ax.legend(fontsize=9)
ax.set_xlim(-5, 10)
ax.set_ylim(-5, 10)
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)

# --- Aperture problem demo ---
ax = axes[1]
edge_img = np.zeros((100, 150), dtype=np.float64)
yy, xx = np.mgrid[0:100, 0:150].astype(np.float64)
edge_img = 0.5 * (1 + np.tanh((xx - 75) / 3.0))  # vertical edge

edge_ix = cv2.Sobel(edge_img, cv2.CV_64F, 1, 0, ksize=5) / 32.0
edge_iy = cv2.Sobel(edge_img, cv2.CV_64F, 0, 1, ksize=5) / 32.0
grad_mag = np.sqrt(edge_ix**2 + edge_iy**2)

ax.imshow(edge_img, cmap="gray", extent=[0, 150, 100, 0])
skip = 10
mask = grad_mag[::skip, ::skip] > 0.01
y_pts = np.arange(0, 100, skip)
x_pts = np.arange(0, 150, skip)
xx_s, yy_s = np.meshgrid(x_pts, y_pts)
gx = edge_ix[::skip, ::skip]
gy = edge_iy[::skip, ::skip]
scale = 200
ax.quiver(xx_s[mask], yy_s[mask], gx[mask], gy[mask],
          color="red", scale=scale, width=0.004)
ax.set_title("Aperture Problem: vertical edge\n(gradient only horizontal → can't detect vertical motion)")
ax.axis("off")

plt.tight_layout()
plt.show()

**Key takeaway**: Each pixel gives one constraint line in $(u, v)$ space. The true flow
lies on every such line (all lines should intersect at the ground-truth flow). But from
a *single pixel alone*, the flow is ambiguous — we need to combine information from
multiple pixels to pin down both $u$ and $v$.

---
## 2. Lucas-Kanade Method

### 2.1 Key Idea: Local Constancy

Lucas & Kanade (1981) resolve the aperture problem by assuming that **all pixels
within a local window $W$ share the same flow** $(u, v)$. If the window is
$n \times n$ (e.g., $15 \times 15 = 225$ pixels), we get **225 equations**
for just **2 unknowns**:

$$
I_{x_i} u + I_{y_i} v = -I_{t_i} \qquad \text{for } i = 1, \dots, n^2
$$

### 2.2 Matrix Formulation

Stack these equations into a matrix system $A \mathbf{d} = \mathbf{b}$:

$$
A = \begin{bmatrix} I_{x_1} & I_{y_1} \\ I_{x_2} & I_{y_2} \\ \vdots & \vdots \\
I_{x_n} & I_{y_n} \end{bmatrix}, \quad
\mathbf{d} = \begin{bmatrix} u \\ v \end{bmatrix}, \quad
\mathbf{b} = \begin{bmatrix} -I_{t_1} \\ -I_{t_2} \\ \vdots \\ -I_{t_n} \end{bmatrix}
$$

### 2.3 Least-Squares Solution (Normal Equations)

Since the system is over-determined, we solve via the **normal equations**:

$$
(A^\top A) \, \mathbf{d} = A^\top \mathbf{b}
$$

$$
\mathbf{d} = (A^\top A)^{-1} A^\top \mathbf{b}
$$

Expanding $A^\top A$:

$$
A^\top A = \begin{bmatrix}
\sum_{i \in W} I_{x_i}^2 & \sum_{i \in W} I_{x_i} I_{y_i} \\
\sum_{i \in W} I_{x_i} I_{y_i} & \sum_{i \in W} I_{y_i}^2
\end{bmatrix}
$$

**This is exactly the structure tensor** $M$ (also called the second-moment matrix)
that appears in Harris corner detection!

> **Note:** In practice, LK uses Gaussian-weighted windows (matching Harris's structure
> tensor exactly). Our from-scratch implementation uses uniform weighting for simplicity.

### 2.4 Connection to Harris & Shi-Tomasi

The matrix $A^\top A$ is well-conditioned (invertible) only when **both eigenvalues**
$\lambda_1, \lambda_2$ are large. This corresponds to:

| Eigenvalues | Region type | LK works? |
|-------------|-------------|----------|
| Both small | Flat region | No — $A^\top A$ is near-singular |
| One large, one small | Edge | No — aperture problem |
| Both large | **Corner / texture** | **Yes!** |

This is why **Shi-Tomasi "Good Features to Track"** selects points where
$\min(\lambda_1, \lambda_2) > \tau$ — these are exactly the points where
Lucas-Kanade can reliably estimate flow. The matrix $A^\top A$ is identical to the
structure tensor $M$ from Harris corner detection (Notebook 05), so **good features
to track = good corners**.

### 2.5 Why Optical Flow Matters for Autonomous Systems

For drones and self-driving vehicles, optical flow is a primary input to:
- **Visual odometry**: estimating ego-motion from sparse feature tracks (Notebook 07)
- **Obstacle avoidance**: divergent flow patterns indicate approaching objects
- **Landing zone assessment**: flow magnitude reveals surface distance and tilt
- **Moving object detection**: flow vectors inconsistent with ego-motion signal independent motion

In [ ]:
def lucas_kanade_point(Ix, Iy, It, px, py, win_size=15):
    """Compute LK optical flow at a single point.

    Parameters
    ----------
    Ix, Iy, It : spatial and temporal gradients
    px, py     : point coordinates (col, row)
    win_size   : window side length (odd)

    Returns
    -------
    u, v : flow at (px, py), or (0, 0) if ill-conditioned
    """
    half = win_size // 2
    H, W = Ix.shape

    y_lo = max(py - half, 0)
    y_hi = min(py + half + 1, H)
    x_lo = max(px - half, 0)
    x_hi = min(px + half + 1, W)

    ix = Ix[y_lo:y_hi, x_lo:x_hi].ravel()
    iy = Iy[y_lo:y_hi, x_lo:x_hi].ravel()
    it = It[y_lo:y_hi, x_lo:x_hi].ravel()

    A = np.column_stack([ix, iy])
    b = -it

    AtA = A.T @ A
    Atb = A.T @ b

    eigvals = np.linalg.eigvalsh(AtA)
    if eigvals.min() < 1e-6:
        return 0.0, 0.0

    d = np.linalg.solve(AtA, Atb)
    return d[0], d[1]

# Test on a single point near a blob center
test_px, test_py = 100, 80
u_est, v_est = lucas_kanade_point(Ix, Iy, It, test_px, test_py, win_size=15)
print(f"LK at ({test_px}, {test_py}): u={u_est:.3f}, v={v_est:.3f}")
print(f"Ground truth:              u={true_dx:.3f}, v={true_dy:.3f}")

In [ ]:
def lucas_kanade_grid(Ix, Iy, It, step=15, win_size=15, min_eig=1e-4):
    """Compute LK flow on a regular grid."""
    H, W = Ix.shape
    half = win_size // 2
    points = []
    flows = []

    for py in range(half, H - half, step):
        for px in range(half, W - half, step):
            u, v = lucas_kanade_point(Ix, Iy, It, px, py, win_size)
            if abs(u) > 1e-8 or abs(v) > 1e-8:
                points.append((px, py))
                flows.append((u, v))

    return np.array(points), np.array(flows)

pts, fls = lucas_kanade_grid(Ix, Iy, It, step=10, win_size=15)

fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(img1, cmap="gray")
if len(pts) > 0:
    mag = np.sqrt(fls[:, 0]**2 + fls[:, 1]**2)
    q = ax.quiver(pts[:, 0], pts[:, 1], fls[:, 0], fls[:, 1],
                  mag, cmap="plasma", scale=50, width=0.003,
                  headwidth=5, headlength=5, headaxislength=4)
    plt.colorbar(q, ax=ax, label="Flow magnitude (px/frame)", shrink=0.8)
ax.set_title(f"Lucas-Kanade Flow Field (GT: u={true_dx}, v={true_dy})", fontsize=13)
ax.axis("off")
plt.tight_layout()
plt.show()

if len(fls) > 0:
    print(f"Mean estimated flow: u={fls[:, 0].mean():.3f}, v={fls[:, 1].mean():.3f}")
    print(f"Std  estimated flow: u={fls[:, 0].std():.3f},  v={fls[:, 1].std():.3f}")

In [ ]:
# --- Compare with OpenCV's pyramidal LK ---

img1_u8 = np.clip(img1 * 255, 0, 255).astype(np.uint8)
img2_u8 = np.clip(img2 * 255, 0, 255).astype(np.uint8)

corners = cv2.goodFeaturesToTrack(img1_u8, maxCorners=200, qualityLevel=0.01,
                                  minDistance=10)

if corners is not None:
    lk_params = dict(winSize=(15, 15), maxLevel=0,
                     criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
    next_pts, status, err = cv2.calcOpticalFlowPyrLK(img1_u8, img2_u8, corners,
                                                     None, **lk_params)
    good_old = corners[status.ravel() == 1].reshape(-1, 2)
    good_new = next_pts[status.ravel() == 1].reshape(-1, 2)
    cv_flow = good_new - good_old

    fig, axes = plt.subplots(1, 2, figsize=(15, 5))
    for ax, title, flow_data, pts_data in [
        (axes[0], "Our LK (from scratch)", fls, pts),
        (axes[1], "OpenCV calcOpticalFlowPyrLK (level=0)", cv_flow,
         good_old.reshape(-1, 2))
    ]:
        ax.imshow(img1, cmap="gray")
        if len(pts_data) > 0:
            mag = np.sqrt(flow_data[:, 0]**2 + flow_data[:, 1]**2)
            q = ax.quiver(pts_data[:, 0], pts_data[:, 1],
                          flow_data[:, 0], flow_data[:, 1],
                          mag, cmap="plasma", scale=50, width=0.003,
                          headwidth=5, headlength=5, headaxislength=4)
            plt.colorbar(q, ax=ax, label="Magnitude (px)", shrink=0.8)
        ax.set_title(title, fontsize=12)
        ax.axis("off")
    plt.suptitle(f"Ground truth flow: ({true_dx}, {true_dy})", fontsize=13)
    plt.tight_layout()
    plt.show()

---
## 3. Pyramidal Lucas-Kanade

### 3.1 The Large-Displacement Problem

The LK derivation relies on a **first-order Taylor expansion**, which is only valid for
small displacements (roughly $|\mathbf{d}| < 1$ pixel at the current scale). When objects
move faster, the linearization breaks down.

### 3.2 Coarse-to-Fine Pyramid

The solution is a **multi-scale pyramid**:

1. Build a Gaussian pyramid: downsample each level by a factor of 2
2. At the **coarsest level** $\ell = L$: large motions become small (a 16-pixel motion
   at the finest level is only 1 pixel at level 4). Apply LK.
3. **Propagate** the estimate to the next finer level: upsample the flow and multiply
   by 2 (since coordinates are doubled).
4. At each finer level: **refine** the residual. Warp the image by the current estimate
   and compute the remaining flow.

### 3.3 Mathematical Formulation

At pyramid level $\ell$:

$$
\mathbf{d}^\ell = \mathbf{d}_{\text{init}}^\ell + \Delta \mathbf{d}^\ell
$$

where the initial guess comes from the coarser level:

$$
\mathbf{d}_{\text{init}}^\ell = 2 \, \mathbf{d}^{\ell+1}
$$

and $\Delta \mathbf{d}^\ell$ is the refinement computed by LK on the **residual** images.

The factor of 2 arises because a displacement of $d$ pixels at level $\ell+1$ corresponds
to $2d$ pixels at level $\ell$ (the image is twice as large).

**Pyramidal Update Rule (Step by Step):**

At pyramid level $L$ (coarsest), start with initial guess $\mathbf{d}^L = \mathbf{0}$. For each level $\ell$ from $L$ down to $0$:

1. **Warp** image $I_2$ at level $\ell$ using the propagated guess: $I_2'(\mathbf{x}) = I_2(\mathbf{x} + \mathbf{d}^{\ell+1}_{\text{upsampled}})$
2. **Compute** spatial gradients $I_x, I_y$ of $I_1$ at level $\ell$
3. **Compute** structure tensor $M = \sum_{\mathbf{w}} \begin{bmatrix} I_x^2 & I_x I_y \\ I_x I_y & I_y^2 \end{bmatrix}$ and temporal gradient $\mathbf{b} = \sum_{\mathbf{w}} \begin{bmatrix} I_x I_t \\ I_y I_t \end{bmatrix}$
4. **Solve** $M \cdot \delta\mathbf{d} = \mathbf{b}$ for the incremental displacement $\delta\mathbf{d}$
5. **Update**: $\mathbf{d}^\ell = 2 \cdot \mathbf{d}^{\ell+1} + \delta\mathbf{d}$

The factor $2$ accounts for the $2\times$ resolution increase between levels. At level $0$, $\mathbf{d}^0$ is the final displacement.

**Key insight:** The pyramid handles large motions by estimating coarse displacement at low resolution (where large motions become small), then refining at higher resolutions.

In [ ]:
def build_pyramid(img, levels=4):
    """Build a Gaussian image pyramid."""
    pyr = [img.copy()]
    for _ in range(levels - 1):
        blurred = gaussian_filter(pyr[-1], sigma=1.0)
        downsampled = blurred[::2, ::2]
        pyr.append(downsampled)
    return pyr

# Visualize a pyramid
pyr = build_pyramid(img1, levels=4)
fig, axes = plt.subplots(1, 4, figsize=(16, 3))
for i, (ax, level_img) in enumerate(zip(axes, pyr)):
    ax.imshow(level_img, cmap="gray")
    ax.set_title(f"Level {i}: {level_img.shape[1]}×{level_img.shape[0]}")
    ax.axis("off")
plt.suptitle("Gaussian Image Pyramid", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)

def make_large_displacement_pair(H=200, W=300, dx=12.0, dy=8.0, n_blobs=10):
    """Blobs with a large displacement that breaks single-level LK."""
    img1 = np.zeros((H, W), dtype=np.float64)
    img2 = np.zeros((H, W), dtype=np.float64)
    yy, xx = np.mgrid[0:H, 0:W].astype(np.float64)

    for _ in range(n_blobs):
        cx = np.random.uniform(40, W - 40)
        cy = np.random.uniform(40, H - 40)
        sigma = np.random.uniform(12, 25)
        amp = np.random.uniform(0.4, 1.0)
        img1 += amp * np.exp(-((xx - cx)**2 + (yy - cy)**2) / (2 * sigma**2))
        img2 += amp * np.exp(-((xx - cx - dx)**2 + (yy - cy - dy)**2) / (2 * sigma**2))

    return np.clip(img1, 0, 1), np.clip(img2, 0, 1), dx, dy

img1_big, img2_big, big_dx, big_dy = make_large_displacement_pair()

Ix_b, Iy_b, It_b = compute_gradients(img1_big, img2_big)
pts_b, fls_b = lucas_kanade_grid(Ix_b, Iy_b, It_b, step=12, win_size=15)

print(f"Large displacement GT: u={big_dx}, v={big_dy}")
if len(fls_b) > 0:
    print(f"Single-level LK mean: u={fls_b[:, 0].mean():.3f}, v={fls_b[:, 1].mean():.3f}")
    print("→ Single-level LK FAILS for large motions!")

In [ ]:
def pyramidal_lk_point(pyr1, pyr2, px, py, win_size=15, n_iter=5):
    """Pyramidal Lucas-Kanade for a single point."""
    levels = len(pyr1)
    u_total, v_total = 0.0, 0.0

    for lev in range(levels - 1, -1, -1):
        scale = 2 ** lev
        px_l = int(round(px / scale))
        py_l = int(round(py / scale))

        H_l, W_l = pyr1[lev].shape
        px_l = np.clip(px_l, 0, W_l - 1)
        py_l = np.clip(py_l, 0, H_l - 1)

        u_init = u_total / scale
        v_init = v_total / scale

        M = np.array([[1, 0, -u_init],
                      [0, 1, -v_init]], dtype=np.float64)
        warped = cv2.warpAffine(pyr2[lev], M, (W_l, H_l),
                                flags=cv2.INTER_LINEAR,
                                borderMode=cv2.BORDER_REFLECT)

        Ix_l, Iy_l, It_l = compute_gradients(pyr1[lev], warped)
        du, dv = lucas_kanade_point(Ix_l, Iy_l, It_l, px_l, py_l, win_size)

        u_total = (u_init + du) * scale
        v_total = (v_init + dv) * scale

    return u_total, v_total

pyr1 = build_pyramid(img1_big, levels=4)
pyr2 = build_pyramid(img2_big, levels=4)

test_px, test_py = 150, 100
u_pyr, v_pyr = pyramidal_lk_point(pyr1, pyr2, test_px, test_py)
print(f"Pyramidal LK at ({test_px},{test_py}): u={u_pyr:.3f}, v={v_pyr:.3f}")
print(f"Ground truth:                        u={big_dx:.3f}, v={big_dy:.3f}")

In [ ]:
def pyramidal_lk_grid(img1, img2, step=12, win_size=15, levels=4):
    """Compute pyramidal LK on a regular grid."""
    pyr1 = build_pyramid(img1, levels)
    pyr2 = build_pyramid(img2, levels)
    H, W = img1.shape
    half = win_size // 2
    points, flows = [], []

    for py in range(half + 5, H - half - 5, step):
        for px in range(half + 5, W - half - 5, step):
            u, v = pyramidal_lk_point(pyr1, pyr2, px, py, win_size)
            points.append((px, py))
            flows.append((u, v))

    return np.array(points), np.array(flows)

pts_pyr, fls_pyr = pyramidal_lk_grid(img1_big, img2_big, step=12)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

axes[0].imshow(img1_big, cmap="gray")
if len(pts_b) > 0:
    mag_b = np.sqrt(fls_b[:, 0]**2 + fls_b[:, 1]**2)
    q0 = axes[0].quiver(pts_b[:, 0], pts_b[:, 1], fls_b[:, 0], fls_b[:, 1],
                         mag_b, cmap="plasma", scale=150, width=0.003,
                         headwidth=5, headlength=5, headaxislength=4)
    plt.colorbar(q0, ax=axes[0], label="Magnitude (px)", shrink=0.8)
axes[0].set_title(f"Single-level LK (fails)\nMean: ({fls_b[:, 0].mean():.1f}, {fls_b[:, 1].mean():.1f})",
                  fontsize=12)
axes[0].axis("off")

axes[1].imshow(img1_big, cmap="gray")
mag_pyr = np.sqrt(fls_pyr[:, 0]**2 + fls_pyr[:, 1]**2)
q1 = axes[1].quiver(pts_pyr[:, 0], pts_pyr[:, 1], fls_pyr[:, 0], fls_pyr[:, 1],
                     mag_pyr, cmap="plasma", scale=150, width=0.003,
                     headwidth=5, headlength=5, headaxislength=4)
plt.colorbar(q1, ax=axes[1], label="Magnitude (px)", shrink=0.8)
axes[1].set_title(f"Pyramidal LK (succeeds)\nMean: ({fls_pyr[:, 0].mean():.1f}, {fls_pyr[:, 1].mean():.1f})",
                  fontsize=12)
axes[1].axis("off")

plt.suptitle(f"Large displacement: GT flow = ({big_dx}, {big_dy})", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
np.random.seed(42)

def make_rotating_pattern(H=200, W=300, n_blobs=15):
    """Create a textured image for rotation tracking."""
    img = np.zeros((H, W), dtype=np.float64)
    yy, xx = np.mgrid[0:H, 0:W].astype(np.float64)
    for _ in range(n_blobs):
        cx = np.random.uniform(20, W - 20)
        cy = np.random.uniform(20, H - 20)
        sigma = np.random.uniform(8, 20)
        amp = np.random.uniform(0.3, 1.0)
        img += amp * np.exp(-((xx - cx)**2 + (yy - cy)**2) / (2 * sigma**2))
    return np.clip(img, 0, 1)

def generate_rotating_sequence(base_img, n_frames=30, angle_step=1.5, tx=1.0, ty=0.5):
    """Generate a sequence with rotation and small translation per frame."""
    H, W = base_img.shape
    center = (W / 2, H / 2)
    frames = [base_img.copy()]

    for i in range(1, n_frames):
        angle = angle_step * i
        M = cv2.getRotationMatrix2D(center, angle, 1.0)
        M[0, 2] += tx * i
        M[1, 2] += ty * i
        rotated = cv2.warpAffine(base_img, M, (W, H),
                                 borderMode=cv2.BORDER_REFLECT)
        frames.append(rotated)

    return frames

base_img = make_rotating_pattern()
frames = generate_rotating_sequence(base_img, n_frames=30)

fig, axes = plt.subplots(1, 5, figsize=(16, 3))
for i, idx in enumerate([0, 7, 14, 21, 29]):
    axes[i].imshow(frames[idx], cmap="gray")
    axes[i].set_title(f"Frame {idx}")
    axes[i].axis("off")
plt.suptitle("Rotating + translating pattern (30 frames)", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
def track_points_pyramidal(frames, initial_points, win_size=15, levels=4):
    """Track points across a sequence of frames using pyramidal LK."""
    n_pts = len(initial_points)
    n_frames = len(frames)
    tracks = np.full((n_frames, n_pts, 2), np.nan)
    tracks[0] = initial_points.copy()
    active = np.ones(n_pts, dtype=bool)

    for f in range(1, n_frames):
        pyr1 = build_pyramid(frames[f - 1], levels)
        pyr2 = build_pyramid(frames[f], levels)
        H, W = frames[f].shape

        for i in range(n_pts):
            if not active[i]:
                continue
            px, py = tracks[f - 1, i]
            if np.isnan(px):
                active[i] = False
                continue
            u, v = pyramidal_lk_point(pyr1, pyr2, int(round(px)), int(round(py)),
                                     win_size)
            new_px = px + u
            new_py = py + v

            margin = 5
            if new_px < margin or new_px >= W - margin or new_py < margin or new_py >= H - margin:
                active[i] = False
                continue

            tracks[f, i] = [new_px, new_py]

    return tracks

frame0_u8 = np.clip(frames[0] * 255, 0, 255).astype(np.uint8)
corners = cv2.goodFeaturesToTrack(frame0_u8, maxCorners=80, qualityLevel=0.02,
                                  minDistance=12)
initial_pts = corners.reshape(-1, 2)
print(f"Tracking {len(initial_pts)} points across {len(frames)} frames...")

tracks = track_points_pyramidal(frames, initial_pts)

survived = np.sum(~np.isnan(tracks[-1, :, 0]))
print(f"{survived} / {len(initial_pts)} points survived to the last frame")

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
ax.imshow(frames[-1], cmap="gray", alpha=0.7)

cmap = plt.cm.jet
n_pts = tracks.shape[1]
for i in range(n_pts):
    valid = ~np.isnan(tracks[:, i, 0])
    if valid.sum() < 2:
        continue
    xs = tracks[valid, i, 0]
    ys = tracks[valid, i, 1]
    color = cmap(i / n_pts)
    ax.plot(xs, ys, "-", color=color, linewidth=1.2, alpha=0.8)
    ax.plot(xs[0], ys[0], "o", color=color, markersize=5)
    ax.plot(xs[-1], ys[-1], "s", color=color, markersize=5)

ax.plot([], [], "o", color="gray", markersize=5, label="Start")
ax.plot([], [], "s", color="gray", markersize=5, label="End")
ax.legend(loc="upper right", fontsize=11, framealpha=0.9)
ax.set_title(f"Point Tracks (Pyramidal LK) — {survived}/{n_pts} survived", fontsize=13)
ax.axis("off")
plt.tight_layout()
plt.show()

---
## 4. Farnebäck Dense Optical Flow

### 4.1 Polynomial Expansion

While Lucas-Kanade computes flow at **sparse** points, many applications (segmentation,
action recognition, video editing) need **dense** flow — a vector for every pixel.

Farnebäck's method approximates each local neighborhood with a **quadratic polynomial**:

$$
f(\mathbf{x}) \approx \mathbf{x}^\top A \mathbf{x} + \mathbf{b}^\top \mathbf{x} + c
$$

where $A$ is a $2 \times 2$ symmetric matrix, $\mathbf{b}$ is a $2 \times 1$ vector,
and $c$ is a scalar. These coefficients are estimated via weighted least squares in a
local window.

### 4.2 Displacement from Two Frames

Given polynomial expansions $f_1(\mathbf{x})$ and $f_2(\mathbf{x})$ from two frames,
assume a global displacement $\mathbf{d}$ such that $f_1(\mathbf{x}) = f_2(\mathbf{x} + \mathbf{d})$.

Substituting:

$$
f_2(\mathbf{x} + \mathbf{d}) = (\mathbf{x} + \mathbf{d})^\top A_2 (\mathbf{x} + \mathbf{d})
+ \mathbf{b}_2^\top (\mathbf{x} + \mathbf{d}) + c_2
$$

Equating the linear terms after expanding and simplifying:

$$
A \mathbf{d} = -\frac{1}{2}(\mathbf{b}_2 - \mathbf{b}_1)
$$

where $A = \frac{1}{2}(A_1 + A_2)$ is the averaged quadratic coefficient matrix.

In practice, Farnebäck solves this in a weighted, iterative, multi-scale scheme.

**Full derivation.** Under displacement $\mathbf{d}$, brightness constancy gives $f_2(\mathbf{x}) \approx f_1(\mathbf{x} - \mathbf{d})$. Expanding:

$$
f_1(\mathbf{x} - \mathbf{d}) = \mathbf{x}^\top A_1 \mathbf{x} + \underbrace{(-2 A_1 \mathbf{d} + \mathbf{b}_1)^\top}_{\text{new linear coeff}} \mathbf{x} + \text{const}
$$

Equating linear coefficients with $f_2$: $\;\mathbf{b}_2 = -2 A_1 \mathbf{d} + \mathbf{b}_1$. Using $A = \tfrac{1}{2}(A_1 + A_2)$ as the averaged quadratic coefficient:

$$
\boxed{\mathbf{d} = -\tfrac{1}{2} A^{-1} (\mathbf{b}_2 - \mathbf{b}_1)}
$$

> **Sign convention.** OpenCV's `calcOpticalFlowFarneback` returns forward flow (frame 1 → frame 2).

### 4.3 Why Farnebäck Remains Relevant

Despite the dominance of learned methods, Farnebäck flow is still widely used in production
autonomous systems for several reasons:

- **Deterministic**: No learned weights means no domain gap, no training-set bias, and
  fully reproducible results. Safety-critical systems benefit from auditable, mathematically
  grounded algorithms.
- **Dense**: Unlike Lucas-Kanade, it produces a flow vector at *every* pixel, which is
  essential for tasks like free-space estimation and occupancy grid updates.
- **CPU-friendly**: Runs at 15–30 fps on a single CPU core at VGA resolution, making it
  viable on embedded platforms without GPU access (e.g., low-power drones, industrial
  inspection robots).
- **No training data required**: Works out of the box on any domain — thermal, infrared,
  medical, underwater — without fine-tuning.

The polynomial expansion model $I(\mathbf{x}) \approx \mathbf{x}^\top A \mathbf{x} + \mathbf{b}^\top \mathbf{x} + c$
captures local image structure through the quadratic matrix $A$ (encoding orientation and
anisotropy) and the linear vector $\mathbf{b}$ (encoding gradient). When the scene displaces
by $\mathbf{d}$, the change in $\mathbf{b}$ between frames directly encodes the displacement
via $\mathbf{d} = -\frac{1}{2} A^{-1}(\mathbf{b}_2 - \mathbf{b}_1)$. This closed-form
solution is solved per-pixel in a weighted least-squares sense, then refined iteratively at
multiple scales — conceptually similar to RAFT's coarse-to-fine updates, but without any
learned components.

### 4.4 HSV Color-Wheel Visualization

The standard way to visualize dense optical flow:
- **Hue** = flow direction ($\arctan(v, u)$)
- **Saturation** = 1 (fully saturated)
- **Value** = flow magnitude (normalized)

This maps the 2D flow vector to a color on the HSV color wheel.

In [ ]:
np.random.seed(42)

def make_multi_motion_scene(H=300, W=400):
    """Create a scene with multiple independently moving rectangular objects."""
    bg_val = 0.2
    img1 = np.full((H, W), bg_val, dtype=np.float64)
    img2 = np.full((H, W), bg_val, dtype=np.float64)
    gt_flow = np.zeros((H, W, 2), dtype=np.float64)

    objects = [
        {"y": 30, "x": 50, "h": 80, "w": 100, "val": 0.9, "dx": 5, "dy": 2},
        {"y": 150, "x": 200, "h": 60, "w": 120, "val": 0.7, "dx": -3, "dy": 4},
        {"y": 80, "x": 250, "h": 90, "w": 70, "val": 0.5, "dx": 0, "dy": -6},
        {"y": 200, "x": 30, "h": 70, "w": 90, "val": 0.6, "dx": 7, "dy": 0},
    ]

    noise = np.random.randn(H, W) * 0.05
    img1 += noise
    img2 += noise

    for obj in objects:
        y, x, h, w = obj["y"], obj["x"], obj["h"], obj["w"]
        dx, dy = obj["dx"], obj["dy"]

        texture = obj["val"] + np.random.randn(h, w) * 0.08

        y1s, y1e = y, y + h
        x1s, x1e = x, x + w
        img1[y1s:y1e, x1s:x1e] = texture

        y2s, y2e = y + dy, y + h + dy
        x2s, x2e = x + dx, x + w + dx
        y2s = np.clip(y2s, 0, H)
        y2e = np.clip(y2e, 0, H)
        x2s = np.clip(x2s, 0, W)
        x2e = np.clip(x2e, 0, W)

        th = y2e - y2s
        tw = x2e - x2s
        if th > 0 and tw > 0:
            img2[y2s:y2e, x2s:x2e] = texture[:th, :tw]

        gt_flow[y1s:y1e, x1s:x1e, 0] = dx
        gt_flow[y1s:y1e, x1s:x1e, 1] = dy

    img1 = np.clip(img1, 0, 1)
    img2 = np.clip(img2, 0, 1)
    return img1, img2, gt_flow

img1_dense, img2_dense, gt_flow = make_multi_motion_scene()

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(img1_dense, cmap="gray", vmin=0, vmax=1)
axes[0].set_title("Frame 1")
axes[1].imshow(img2_dense, cmap="gray", vmin=0, vmax=1)
axes[1].set_title("Frame 2")
axes[2].imshow(np.abs(img2_dense - img1_dense), cmap="hot")
axes[2].set_title("|Difference|")
for ax in axes:
    ax.axis("off")
plt.suptitle("Multi-motion synthetic scene", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
def flow_to_color(flow, max_mag=None):
    """Convert optical flow to RGB via HSV color wheel (Middlebury convention).

    Hue  = flow direction, Saturation = 1, Value = flow magnitude.
    """
    u = flow[..., 0]
    v = flow[..., 1]
    mag = np.sqrt(u**2 + v**2)
    ang = np.arctan2(v, u)

    if max_mag is None:
        max_mag = mag.max() + 1e-8

    h = (ang + np.pi) / (2 * np.pi)  # [0, 1]
    s = np.ones_like(h)
    v_ch = np.clip(mag / max_mag, 0, 1)

    hsv = np.stack([h, s, v_ch], axis=-1)
    rgb = hsv_to_rgb(hsv)
    return rgb


def make_color_wheel_legend(size=200):
    """Create a color wheel legend for flow visualization."""
    y, x = np.mgrid[-size//2:size//2, -size//2:size//2].astype(np.float64)
    r = np.sqrt(x**2 + y**2)
    theta = np.arctan2(y, x)

    h = (theta + np.pi) / (2 * np.pi)
    s = np.ones_like(h)
    v = np.clip(r / (size // 2), 0, 1)

    mask = r <= size // 2
    hsv = np.stack([h, s * mask, v * mask], axis=-1)
    rgb = hsv_to_rgb(hsv)
    rgb[~mask] = 1.0
    return rgb

In [ ]:
img1_u8_d = np.clip(img1_dense * 255, 0, 255).astype(np.uint8)
img2_u8_d = np.clip(img2_dense * 255, 0, 255).astype(np.uint8)

farneback_flow = cv2.calcOpticalFlowFarneback(
    img1_u8_d, img2_u8_d, None,
    pyr_scale=0.5, levels=3, winsize=15,
    iterations=3, poly_n=5, poly_sigma=1.2, flags=0
)

max_mag = max(np.sqrt(gt_flow[..., 0]**2 + gt_flow[..., 1]**2).max(),
              np.sqrt(farneback_flow[..., 0]**2 + farneback_flow[..., 1]**2).max())

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

axes[0].imshow(flow_to_color(gt_flow, max_mag))
axes[0].set_title("Ground Truth Flow (HSV)")
axes[0].axis("off")

axes[1].imshow(flow_to_color(farneback_flow, max_mag))
axes[1].set_title("Farnebäck Flow (HSV)")
axes[1].axis("off")

wheel = make_color_wheel_legend(size=150)
axes[2].imshow(wheel)
axes[2].set_title("Color Wheel Legend")
axes[2].annotate("→", xy=(0.95, 0.5), xycoords="axes fraction", fontsize=14, ha="center")
axes[2].annotate("←", xy=(0.05, 0.5), xycoords="axes fraction", fontsize=14, ha="center")
axes[2].annotate("↑", xy=(0.5, 0.05), xycoords="axes fraction", fontsize=14, ha="center")
axes[2].annotate("↓", xy=(0.5, 0.95), xycoords="axes fraction", fontsize=14, ha="center")
axes[2].axis("off")

plt.suptitle("Dense Optical Flow — Farnebäck Method", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(img1_dense, cmap="gray", alpha=0.6)

step = 12
H, W = farneback_flow.shape[:2]
y_pts = np.arange(step // 2, H, step)
x_pts = np.arange(step // 2, W, step)
xx_q, yy_q = np.meshgrid(x_pts, y_pts)
u_q = farneback_flow[yy_q, xx_q, 0]
v_q = farneback_flow[yy_q, xx_q, 1]
mag_q = np.sqrt(u_q**2 + v_q**2)

q = ax.quiver(xx_q, yy_q, u_q, v_q, mag_q, cmap="plasma",
              scale=80, width=0.003, headwidth=4)
plt.colorbar(q, ax=ax, label="Flow magnitude (px)")
ax.set_title("Farnebäck Dense Flow — Quiver Plot", fontsize=13)
ax.axis("off")
plt.tight_layout()
plt.show()

---
## 5. RAFT — Recurrent All-Pairs Field Transforms

### 5.1 Architecture Overview

RAFT (Teed & Deng, ECCV 2020) represents a paradigm shift in optical flow estimation.
Instead of the classical coarse-to-fine pyramid, RAFT uses:

1. **Feature Encoder**: A CNN (ResNet-like) extracts dense per-pixel feature maps
   $\mathbf{f}_1, \mathbf{f}_2 \in \mathbb{R}^{H/8 \times W/8 \times D}$ from both frames.

2. **Correlation Volume**: Compute the **4D all-pairs correlation** between the two
   feature maps:
   $$C_{ijkl} = \langle \mathbf{f}_1(i, j), \, \mathbf{f}_2(k, l) \rangle$$
   This has shape $(H/8 \times W/8 \times H/8 \times W/8)$ — it captures the similarity
   between every pair of pixels.

3. **Correlation Pyramid**: Average-pool the last two dimensions of $C$ at multiple
   scales to capture both fine and coarse correspondences.

4. **Iterative Update Operator**: A ConvGRU (gated recurrent unit) that takes:
   - Current flow estimate $\mathbf{f}_k$
   - Correlation features (looked up from the pyramid at the current flow location)
   - Context features from frame 1
   
   and outputs a flow update $\Delta \mathbf{f}$.

5. **Iterative Refinement**: Start from **zero flow**, then iterate 12–32 times:
   $$\mathbf{f}_{k+1} = \mathbf{f}_k + \Delta \mathbf{f}_k$$

### 5.2 Training Loss

RAFT is trained with supervision on all iterations, with exponentially increasing
weights:

$$
L = \sum_{i=1}^{N} \gamma^{N-i} \, \| \mathbf{f}_i - \mathbf{f}_{\text{gt}} \|_1
$$

where $\gamma = 0.8$. This encourages all iterations to produce good flow, with
later iterations weighted more heavily.

### 5.3 Correlation Lookup

At iteration $k$, the current flow estimate $\mathbf{f}_k(i,j)$ defines a predicted correspondence location $(i + u_k, j + v_k)$ in frame 2. RAFT looks up a local $r \times r$ grid (typically $r = 4$) around this location in each level of the correlation pyramid:

$$
\text{corr}^{(l)}_k(i,j,\delta) = C^{(l)}_{ij,\, \lfloor(i+u_k)/2^l\rfloor + \delta_x,\, \lfloor(j+v_k)/2^l\rfloor + \delta_y}
\quad \text{for } \delta \in \{-r, \ldots, r\}^2
$$

Using bilinear interpolation for sub-pixel locations. The multi-scale lookup provides both fine detail (level 0) and large-displacement context (higher levels) at constant cost: $4 \times (2r+1)^2$ values per pixel regardless of image size.

### 5.4 ConvGRU Update Operator

The update operator is a convolutional GRU with gates:

$$
z_t = \sigma(W_z * [\text{corr}_k, \mathbf{f}_k, \text{ctx}] + b_z) \quad \text{(update gate)}
$$
$$
r_t = \sigma(W_r * [\text{corr}_k, \mathbf{f}_k, \text{ctx}] + b_r) \quad \text{(reset gate)}
$$
$$
\tilde{h}_t = \tanh(W_h * [\text{corr}_k, r_t \odot h_{t-1}, \text{ctx}] + b_h)
$$
$$
h_t = (1 - z_t) \odot h_{t-1} + z_t \odot \tilde{h}_t
$$

Two linear heads predict $\Delta \mathbf{f}$ and a confidence mask $\mathbf{m}$ from $h_t$.

### 5.5 Convex Upsampling

RAFT operates at 1/8 resolution. Rather than bilinear upsampling (blurs edges), RAFT predicts a **convex combination** mask $\mathbf{m} \in \mathbb{R}^{H/8 \times W/8 \times 8 \times 8 \times 9}$ from the GRU hidden state. Each full-resolution pixel $(i,j)$ is a weighted sum of its $3\times 3$ coarse neighbors:

$$
\mathbf{f}^{\text{full}}(i,j) = \sum_{(p,q) \in \mathcal{N}_{3 \times 3}} m_{pq}(i,j) \cdot \mathbf{f}^{\text{coarse}}(p,q), \quad \sum_{p,q} m_{pq} = 1
$$

The weights are produced by a softmax, ensuring convexity (preserves flow edges).

### 5.6 Why RAFT Works So Well

- **Global context**: The all-pairs correlation volume captures correspondences
  at *every* displacement, unlike LK which only sees local windows.
- **Iterative refinement**: Instead of making one large prediction, RAFT makes
  many small corrections — more stable and robust.
- **No coarse-to-fine**: Traditional multi-scale approaches can lose small, fast-moving
  objects at coarse levels. RAFT operates at a single resolution.
- **State-of-the-art**: Dramatically improved accuracy on Sintel and KITTI benchmarks.

### 5.7 SEA-RAFT and Uncertainty Estimation (Wang et al., ECCV 2024)

**SEA-RAFT** (Simple, Efficient, Accurate RAFT) introduced three key improvements:

1. **Direct flow initialization**: Instead of starting from zero flow, SEA-RAFT
   predicts an initial flow from the correlation volume, reducing the number of
   refinement iterations needed (6 vs 32 for equivalent accuracy → **2.3× faster**)

2. **Mixture-of-Laplace (MoL) loss**: Replaces the standard $L_1$ loss with a
   probabilistic model that outputs per-pixel **uncertainty**:
   $$p(\mathbf{f}_{\text{gt}} | \mathbf{f}, b, \alpha) = \alpha \cdot \text{Lap}(\mathbf{f}, 0) + (1 - \alpha) \cdot \text{Lap}(\mathbf{f}, b)$$
   where $b$ is a learned scale parameter. The first component (fixed $\beta_1 = 0$)
   optimizes for EPE accuracy; the second handles ambiguous regions (occlusions,
   reflections). This gives **calibrated uncertainty maps for free**.

3. **Rigid-motion pre-training**: Pre-training on synthetically rendered rigid
   scenes improves cross-dataset generalization (e.g., Sintel → KITTI).

**Why uncertainty matters for autonomous systems:** The predicted uncertainty from
SEA-RAFT can be propagated into downstream VIO/SLAM systems as measurement
covariances, so that unreliable flow in occluded or textureless regions is
automatically down-weighted during state estimation.

In [ ]:
fig, ax = plt.subplots(figsize=(14, 7))
ax.set_xlim(0, 14)
ax.set_ylim(0, 8)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("RAFT Architecture Overview", fontsize=16, fontweight="bold", pad=15)

import matplotlib.patches as mpatches

blocks = [
    {"xy": (0.3, 5.5), "w": 2.0, "h": 1.8, "label": "Frame 1\n$I_1$", "color": "#4CAF50"},
    {"xy": (0.3, 2.5), "w": 2.0, "h": 1.8, "label": "Frame 2\n$I_2$", "color": "#2196F3"},
    {"xy": (3.5, 5.5), "w": 2.2, "h": 1.8, "label": "Feature\nEncoder\n$\mathbf{f}_1$",
     "color": "#FF9800"},
    {"xy": (3.5, 2.5), "w": 2.2, "h": 1.8, "label": "Feature\nEncoder\n$\mathbf{f}_2$",
     "color": "#FF9800"},
    {"xy": (7.0, 3.8), "w": 2.2, "h": 2.2, "label": "Correlation\nVolume\n$C_{ijkl}$",
     "color": "#9C27B0"},
    {"xy": (10.2, 3.8), "w": 2.5, "h": 2.2, "label": "ConvGRU\nUpdate\nOperator",
     "color": "#F44336"},
]

for b in blocks:
    rect = mpatches.FancyBboxPatch(b["xy"], b["w"], b["h"],
                                    boxstyle="round,pad=0.1",
                                    facecolor=b["color"], alpha=0.3,
                                    edgecolor=b["color"], linewidth=2)
    ax.add_patch(rect)
    ax.text(b["xy"][0] + b["w"]/2, b["xy"][1] + b["h"]/2, b["label"],
            ha="center", va="center", fontsize=10, fontweight="bold")

arrows = [
    ((2.3, 6.4), (3.5, 6.4)),
    ((2.3, 3.4), (3.5, 3.4)),
    ((5.7, 6.0), (7.0, 5.2)),
    ((5.7, 3.8), (7.0, 4.5)),
    ((9.2, 4.9), (10.2, 4.9)),
]
for (x1, y1), (x2, y2) in arrows:
    ax.annotate("", xy=(x2, y2), xytext=(x1, y1),
                arrowprops=dict(arrowstyle="->", lw=2, color="#333"))

ax.annotate("", xy=(11.45, 3.8), xytext=(11.45, 2.2),
            arrowprops=dict(arrowstyle="->", lw=2, color="#333",
                            connectionstyle="arc3,rad=-0.3"))
ax.text(12.5, 2.8, "Iterate\n12–32×", fontsize=10, style="italic",
        ha="center", va="center")

ax.add_patch(mpatches.FancyBboxPatch((10.0, 0.8), 3.0, 1.2,
              boxstyle="round,pad=0.1", facecolor="#E8F5E9",
              edgecolor="#4CAF50", linewidth=2))
ax.text(11.5, 1.4, "Flow output\n$\mathbf{f}_{final}$",
        ha="center", va="center", fontsize=11, fontweight="bold")

ax.text(7, 1.0, r"$L = \sum_{i=1}^{N} \gamma^{N-i} \| \mathbf{f}_i - \mathbf{f}_{gt} \|_1$"
        f"\n($\\gamma = 0.8$)",
        fontsize=12, ha="center", va="center",
        bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

plt.tight_layout()
plt.show()

In [ ]:
print("""
RAFT Pseudocode
===============

def raft_forward(I1, I2, iters=12):
    # 1. Extract features
    f1 = feature_encoder(I1)          # (B, D, H/8, W/8)
    f2 = feature_encoder(I2)          # (B, D, H/8, W/8)
    ctx = context_encoder(I1)         # (B, D_ctx, H/8, W/8)

    # 2. Build 4D correlation volume
    C = einsum('bdhw, bdij -> bhwij', f1, f2)   # (B, H/8, W/8, H/8, W/8)

    # 3. Build correlation pyramid (pool last 2 dims)
    corr_pyramid = [C]
    for level in range(3):
        C_pooled = avg_pool_2d(corr_pyramid[-1], kernel=2)  # over (i,j) dims
        corr_pyramid.append(C_pooled)

    # 4. Initialize flow at zero
    flow = zeros(B, 2, H/8, W/8)
    hidden = zeros(B, D_hidden, H/8, W/8)

    flow_predictions = []

    # 5. Iterative refinement
    for k in range(iters):
        # Look up correlation at current flow estimate
        corr_features = lookup_correlation(corr_pyramid, flow)

        # GRU update
        hidden, delta_flow = update_block(hidden, ctx, corr_features, flow)
        flow = flow + delta_flow

        # Upsample to full resolution
        flow_up = upsample_flow(flow, mask=None)  # convex upsampling
        flow_predictions.append(flow_up)

    return flow_predictions
""")

---
## 6. Point Tracking — Beyond Two-Frame Flow

### 6.1 Motivation

Optical flow estimates motion between **two consecutive frames**. But many applications
need to track the same point across an **entire video** — tens, hundreds, or thousands
of frames. Naively chaining two-frame flow leads to **drift** (errors accumulate).

Modern point trackers address this with learned temporal models.

### 6.2 TAPIR — Tracking Any Point with per-frame Initialization and temporal Refinement

TAPIR (Doersch et al., ICCV 2023) works in two stages:

**Stage 1 — Matching**: For each query point, compute a cost volume against every
frame to get a coarse location estimate. This provides **per-frame initialization**
that avoids drift.

**Stage 2 — Refinement**: A temporal network (1D convolution over time) smooths
the trajectory and predicts:
- Refined $(x, y)$ position
- **Visibility flag**: is the point occluded in this frame?
- Uncertainty estimate

Key properties:
- Can track **any** query point (not just detected features)
- Handles **occlusions** explicitly
- Works on **any video** (not just specific categories)

### 6.3 CoTracker — Track Together

CoTracker (Karaev et al., ECCV 2024) takes a different approach: track **multiple
points jointly** using a transformer.

Key insight: Points can **help each other**. If one point is occluded, nearby visible
points provide context for where it should be.

Architecture:
- Correlation features for each point (similar to RAFT)
- Transformer attention **across points** (not just across time)
- Sliding window for long videos
- Can track **forward and backward** in time

### 6.4 Challenges in Long-Range Tracking

| Challenge | Description |
|-----------|-------------|
| Occlusion | Points disappear behind other objects |
| Drift | Small errors accumulate over many frames |
| Appearance change | Lighting, viewpoint, deformation |
| Fast motion | Points move many pixels between frames |
| Scale change | Objects approach or recede from camera |

### 6.5 Applications

- **Video editing**: propagate edits across frames
- **3D reconstruction**: long-range correspondences improve SfM
- **Action recognition**: motion patterns from tracked points
- **Robotics**: track manipulation targets

### 6.6 Dense Point Tracking as Long-Range Optical Flow

A key conceptual insight has emerged: **dense point tracking is simply long-range optical flow**.
Classical optical flow estimates a displacement field between two consecutive frames;
dense point tracking estimates that same displacement field across an *entire video*.
Every pixel gets a trajectory, not just sparse query points. This reframing unifies
two historically separate problems and has driven a wave of architectures that excel at both.

### 6.7 SPOT — Streaming Online Dense Point Tracking (ICCV 2025)

SPOT (Li et al., ICCV 2025) introduces a **streaming memory architecture** that processes
frames online — one at a time, in order — making it suitable for real-time applications.

Three core components:

1. **Memory reading** — enriches the current frame's features by attending to a compact
   memory bank of past observations, providing long-range context without storing every frame.
2. **Sensory memory** — a lightweight recurrent state that captures **short-term dynamics**
   (velocity, acceleration) for smooth trajectory prediction between memory reads.
3. **Visibility-guided splatting** — propagates tracking information from visible points
   to newly visible regions using differentiable splatting, weighted by predicted visibility
   scores. This avoids the information loss that occurs when tracked points become occluded
   and later reappear.

**Efficiency:** SPOT runs **2× faster** than prior state-of-the-art methods while using
**10× fewer parameters**, making it the first dense point tracker practical for edge deployment.

### 6.8 AllTracker — Dense Tracking as Long-Range Flow (ICCV 2025)

AllTracker (Wang et al., ICCV 2025) explicitly frames dense point tracking as **long-range
dense optical flow** and builds an architecture around this idea.

Architecture:
- **Spatial 2D convolution** captures local appearance and structure within each frame
- **Temporal pixel-aligned attention** models correspondences across time without
  expensive global attention over the full spatio-temporal volume
- This factored design scales to high-resolution, long videos

**Critical finding:** Joint training on *both* optical flow datasets (Sintel, KITTI) and
point tracking datasets (Kubric, TAP-Vid) is essential for top performance. Neither data
source alone suffices — flow data provides dense sub-pixel supervision on real scenes, while
tracking data provides long-range temporal reasoning. This confirms that the two tasks share
a common correspondence backbone.

### 6.9 The Unification Trend

Optical flow and point tracking are **converging**. Both are instances of the same underlying
problem: **dense visual correspondence estimation**. They differ only in temporal scope:

| Aspect | Classical Optical Flow | Dense Point Tracking |
|--------|----------------------|---------------------|
| Temporal scope | 2 consecutive frames | Arbitrary frame pairs |
| Output | Per-pixel displacement field | Per-pixel trajectory |
| Occlusion handling | Rarely modeled | Explicit visibility prediction |
| Training data | Flow-specific (Sintel, KITTI) | Tracking-specific (Kubric) |

The trend is clear: modern architectures like AllTracker and SPOT can be trained on *both*
kinds of data and evaluated on *both* benchmarks. We should expect future systems to treat
"optical flow" and "point tracking" as two evaluation protocols for a single model, not as
separate tasks requiring separate architectures.

In [ ]:
print("""
API Usage Patterns for Modern Point Trackers
=============================================

# --- TAPIR ---
import torch
from tapnet.torch import tapir_model

model = tapir_model.TAPIR(checkpoint='tapir_checkpoint.pt')
model.eval()

# video: (1, T, H, W, 3) float32 in [0, 1]
# query_points: (1, N, 3) — each row is (t, y, x) where t is the query frame
outputs = model(video, query_points)
# outputs['tracks']:    (1, N, T, 2)  — (x, y) positions
# outputs['occlusion']: (1, N, T)     — occlusion logits
# outputs['expected_dist']: (1, N, T) — uncertainty

# --- CoTracker ---
from cotracker.predictor import CoTrackerPredictor

model = CoTrackerPredictor(checkpoint='cotracker2.pth')
# video: (1, T, 3, H, W) float32
# queries: (1, N, 3) — each row is (t, x, y)
pred_tracks, pred_visibility = model(video, queries=queries)
# pred_tracks:     (1, T, N, 2)  — (x, y) positions
# pred_visibility: (1, T, N)     — boolean visibility
""")

In [ ]:
np.random.seed(42)

def simulate_point_tracks_with_occlusion(n_points=12, n_frames=60, H=300, W=400):
    """Simulate point tracks with occlusion events."""
    tracks = np.zeros((n_frames, n_points, 2))
    visible = np.ones((n_frames, n_points), dtype=bool)

    for i in range(n_points):
        x0 = np.random.uniform(50, W - 50)
        y0 = np.random.uniform(50, H - 50)
        vx = np.random.uniform(-2, 2)
        vy = np.random.uniform(-1.5, 1.5)

        occ_start = np.random.randint(15, 40)
        occ_dur = np.random.randint(5, 15)

        for t in range(n_frames):
            noise_x = np.random.randn() * 0.3
            noise_y = np.random.randn() * 0.3
            x = x0 + vx * t + 10 * np.sin(0.1 * t) + noise_x
            y = y0 + vy * t + 5 * np.cos(0.15 * t) + noise_y
            tracks[t, i] = [x, y]

            if occ_start <= t < occ_start + occ_dur:
                visible[t, i] = False

    return tracks, visible

tracks_sim, vis_sim = simulate_point_tracks_with_occlusion()

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

ax = axes[0]
for i in range(tracks_sim.shape[1]):
    xs = tracks_sim[:, i, 0]
    ys = tracks_sim[:, i, 1]
    v = vis_sim[:, i]
    color = plt.cm.tab10(i / tracks_sim.shape[1])
    ax.plot(xs[v], ys[v], "o-", color=color, markersize=2, linewidth=1, alpha=0.8)
    ax.plot(xs[~v], ys[~v], "x", color=color, markersize=5, alpha=0.4)
    ax.plot(xs[0], ys[0], "o", color=color, markersize=8, zorder=5)
ax.set_xlim(0, 400)
ax.set_ylim(300, 0)
ax.set_title("Simulated Point Tracks\n(circles = visible, × = occluded)", fontsize=12)
ax.set_xlabel("x (px)")
ax.set_ylabel("y (px)")
ax.grid(True, alpha=0.3)

ax = axes[1]
for i in range(tracks_sim.shape[1]):
    v = vis_sim[:, i].astype(float)
    ax.plot(range(len(v)), v + i * 1.2, color=plt.cm.tab10(i / tracks_sim.shape[1]),
            linewidth=3)
ax.set_xlabel("Frame", fontsize=11)
ax.set_ylabel("Point index", fontsize=11)
ax.set_title("Visibility Timeline\n(high = visible, low = occluded)", fontsize=12)
ax.set_yticks([])
ax.grid(True, axis="x", alpha=0.3)

plt.tight_layout()
plt.show()

---
## 7. Scene Flow — 3D Motion Vectors

### 7.1 From 2D Flow to 3D Motion

**Scene flow** extends optical flow into 3D: instead of a 2D velocity at each pixel,
we recover the full **3D motion vector** $\Delta \mathbf{P} \in \mathbb{R}^3$.

Given:
- 2D optical flow $(u, v)$ between frames
- Depth maps $Z_1$ (frame 1) and $Z_2$ (frame 2)
- Camera intrinsic matrix $K$

### 7.2 Derivation

**Step 1**: Back-project pixel $(x, y)$ from frame 1 to 3D using depth $Z_1$:

$$
\mathbf{P}_1 = Z_1(x, y) \cdot K^{-1} \begin{bmatrix} x \\ y \\ 1 \end{bmatrix}
$$

Expanding with a standard pinhole camera:

$$
K = \begin{bmatrix} f_x & 0 & c_x \\ 0 & f_y & c_y \\ 0 & 0 & 1 \end{bmatrix}
\quad \Rightarrow \quad
\mathbf{P}_1 = Z_1 \begin{bmatrix} (x - c_x) / f_x \\ (y - c_y) / f_y \\ 1 \end{bmatrix}
$$

**Step 2**: The corresponding pixel in frame 2 is at $(x + u, y + v)$. Back-project
using $Z_2$:

$$
\mathbf{P}_2 = Z_2(x + u, y + v) \cdot K^{-1} \begin{bmatrix} x + u \\ y + v \\ 1 \end{bmatrix}
$$

**Step 3**: The **scene flow** at pixel $(x, y)$ is the 3D displacement:

$$
\boxed{\text{Scene Flow}(x, y) = \mathbf{P}_2 - \mathbf{P}_1}
$$

This gives us a 3D velocity vector at each pixel, telling us how the corresponding
3D point has moved.

### 7.3 Why Scene Flow Matters for Autonomous Systems

Scene flow is the bridge between **perception** and **prediction** — it tells the
autonomous system not just *where* obstacles are, but *how they are moving* in 3D.

**Error analysis**: Scene flow inherits errors from *both* depth estimation and
optical flow.  If $\sigma_Z$ is the depth noise and $\sigma_f$ is the flow noise:

$$\sigma_{\text{SF}} \approx \sqrt{2\sigma_Z^2 + \left(\frac{Z}{f}\right)^2 \sigma_f^2}$$

At 5 m depth with $f = 500$ px, a 1 px flow error produces a 1 cm lateral velocity
error — acceptable for obstacle avoidance but not for precision manipulation.

### 7.4 Applications

- **Autonomous driving**: Identify which objects are moving and how fast
  (critical for trajectory prediction)
- **Robotics**: Estimate object motion for grasping and manipulation
- **Dynamic reconstruction**: Separate static and dynamic scene parts
- **Action recognition**: 3D motion cues from depth cameras

In [ ]:
np.random.seed(42)

H, W = 100, 150
fx, fy = 200.0, 200.0
cx, cy = W / 2, H / 2
K = np.array([[fx, 0, cx],
              [0, fy, cy],
              [0,  0,  1]], dtype=np.float64)
K_inv = np.linalg.inv(K)

yy, xx = np.mgrid[0:H, 0:W].astype(np.float64)

depth1 = 5.0 + 0.5 * np.sin(xx * 0.05) + 0.3 * np.cos(yy * 0.07)
depth1 += np.random.randn(H, W) * 0.05

flow_2d = np.zeros((H, W, 2), dtype=np.float64)

obj1_mask = (xx > 30) & (xx < 70) & (yy > 20) & (yy < 60)
flow_2d[obj1_mask, 0] = 4.0   # moving right
flow_2d[obj1_mask, 1] = 1.0   # slightly down

obj2_mask = (xx > 80) & (xx < 130) & (yy > 40) & (yy < 80)
flow_2d[obj2_mask, 0] = -2.0  # moving left
flow_2d[obj2_mask, 1] = -3.0  # moving up

depth1[obj1_mask] = 3.0
depth1[obj2_mask] = 4.0

depth2 = depth1.copy()
obj1_displaced = ((xx - 4) > 30) & ((xx - 4) < 70) & ((yy - 1) > 20) & ((yy - 1) < 60)
obj2_displaced = ((xx + 2) > 80) & ((xx + 2) < 130) & ((yy + 3) > 40) & ((yy + 3) < 80)
depth2[obj1_displaced] = 3.0 - 0.3  # approaching
depth2[obj2_displaced] = 4.0 + 0.2  # receding

print(f"Image size: {W}×{H}")
print(f"Intrinsics: fx={fx}, fy={fy}, cx={cx}, cy={cy}")
print(f"Depth range: [{depth1.min():.2f}, {depth1.max():.2f}]")

In [ ]:
def compute_scene_flow(depth1, depth2, flow_2d, K_inv):
    """Compute 3D scene flow from depth maps and 2D optical flow.

    Parameters
    ----------
    depth1, depth2 : (H, W) depth maps for frames 1 and 2
    flow_2d        : (H, W, 2) optical flow [u, v]
    K_inv          : (3, 3) inverse camera intrinsic matrix

    Returns
    -------
    scene_flow : (H, W, 3) 3D motion vectors [dX, dY, dZ]
    P1         : (H, W, 3) 3D points in frame 1
    P2         : (H, W, 3) 3D points in frame 2
    """
    H, W = depth1.shape
    yy, xx = np.mgrid[0:H, 0:W].astype(np.float64)

    ones = np.ones((H, W), dtype=np.float64)
    pixels1 = np.stack([xx, yy, ones], axis=-1)  # (H, W, 3)
    rays1 = np.einsum("ij,hwj->hwi", K_inv, pixels1)  # (H, W, 3)
    P1 = depth1[..., None] * rays1

    xx2 = xx + flow_2d[..., 0]
    yy2 = yy + flow_2d[..., 1]
    pixels2 = np.stack([xx2, yy2, ones], axis=-1)
    rays2 = np.einsum("ij,hwj->hwi", K_inv, pixels2)

    xx2_int = np.clip(np.round(xx2).astype(int), 0, W - 1)
    yy2_int = np.clip(np.round(yy2).astype(int), 0, H - 1)
    depth2_sampled = depth2[yy2_int, xx2_int]

    P2 = depth2_sampled[..., None] * rays2

    scene_flow = P2 - P1
    return scene_flow, P1, P2

sf, P1, P2 = compute_scene_flow(depth1, depth2, flow_2d, K_inv)
sf_mag = np.sqrt(np.sum(sf**2, axis=-1))

print(f"Scene flow shape: {sf.shape}")
print(f"Max 3D motion magnitude: {sf_mag.max():.4f}")
print(f"Mean 3D motion (moving obj 1): dX={sf[obj1_mask, 0].mean():.4f}, "
      f"dY={sf[obj1_mask, 1].mean():.4f}, dZ={sf[obj1_mask, 2].mean():.4f}")
print(f"Mean 3D motion (moving obj 2): dX={sf[obj2_mask, 0].mean():.4f}, "
      f"dY={sf[obj2_mask, 1].mean():.4f}, dZ={sf[obj2_mask, 2].mean():.4f}")

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(18, 4))

im0 = axes[0].imshow(depth1, cmap="viridis")
axes[0].set_title("Depth map (frame 1)")
plt.colorbar(im0, ax=axes[0], shrink=0.8)

flow_mag_2d = np.sqrt(flow_2d[..., 0]**2 + flow_2d[..., 1]**2)
im1 = axes[1].imshow(flow_mag_2d, cmap="hot")
axes[1].set_title("2D flow magnitude")
plt.colorbar(im1, ax=axes[1], shrink=0.8)

im2 = axes[2].imshow(sf_mag, cmap="hot")
axes[2].set_title("3D scene flow magnitude")
plt.colorbar(im2, ax=axes[2], shrink=0.8)

im3 = axes[3].imshow(sf[..., 2], cmap="RdBu_r")
axes[3].set_title("Scene flow Z-component\n(blue=approaching, red=receding)")
plt.colorbar(im3, ax=axes[3], shrink=0.8)

for ax in axes:
    ax.axis("off")

plt.suptitle("Scene Flow Components", fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
fig = plt.figure(figsize=(18, 7))

step = 4
xs = P1[::step, ::step, 0].ravel()
ys = P1[::step, ::step, 1].ravel()
zs = P1[::step, ::step, 2].ravel()
us = sf[::step, ::step, 0].ravel()
vs = sf[::step, ::step, 1].ravel()
ws = sf[::step, ::step, 2].ravel()
mags = np.sqrt(us**2 + vs**2 + ws**2)

moving = mags > 0.01

# --- Left: 3D scene flow with quiver arrows ---
ax1 = fig.add_subplot(121, projection="3d")
ax1.scatter(xs[~moving], ys[~moving], zs[~moving],
            c=zs[~moving], cmap="Blues", s=1, alpha=0.15, label="Static scene")

if moving.any():
    q = ax1.quiver(xs[moving], ys[moving], zs[moving],
                   us[moving], vs[moving], ws[moving],
                   length=2.0, normalize=False, arrow_length_ratio=0.3,
                   linewidth=1.8)
    q.set_array(mags[moving])
    q.set_cmap("plasma")
    cbar = fig.colorbar(q, ax=ax1, shrink=0.5, pad=0.08)
    cbar.set_label("3D flow magnitude [m]", fontsize=10)

ax1.set_xlabel("X [m]"); ax1.set_ylabel("Y [m]"); ax1.set_zlabel("Z [m]")
ax1.set_title("3D Scene Flow — Moving Objects", fontsize=13)
ax1.view_init(elev=25, azim=-60)
ax1.legend(loc="upper left", fontsize=9)

# --- Right: Motion segmentation (top-down projection) ---
ax2 = fig.add_subplot(122)
sf_mag_2d = np.sqrt(np.sum(sf**2, axis=-1))
im = ax2.imshow(sf_mag_2d, cmap="hot", interpolation="nearest")
plt.colorbar(im, ax=ax2, label="3D motion magnitude [m]")
ax2.set_title("Motion Segmentation from Scene Flow", fontsize=13)
ax2.set_xlabel("Pixel x"); ax2.set_ylabel("Pixel y")
contour_mask = (sf_mag_2d > 0.01).astype(float)
ax2.contour(contour_mask, levels=[0.5], colors='cyan', linewidths=1.5)

plt.suptitle("Scene Flow: 2D → 3D Motion Recovery", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

---
## 8. Exercises

### Exercise 1: Implement Lucas-Kanade for a Single Point

Fill in the `TODO` sections to implement the Lucas-Kanade optical flow algorithm
for a single point. This reinforces the normal-equations derivation from Section 2.

In [ ]:
def exercise_lk_single_point(Ix, Iy, It, px, py, win_size=15):
    """Lucas-Kanade optical flow at a single point.

    Parameters
    ----------
    Ix, Iy, It : (H, W) spatial and temporal image gradients
    px, py     : pixel coordinates (column, row)
    win_size   : side length of the local window (odd integer)

    Returns
    -------
    u, v : estimated flow components
    """
    half = win_size // 2
    H, W = Ix.shape

    # 1. Extract window (clip at image boundaries)
    y_lo = max(0, py - half)
    y_hi = min(H, py + half + 1)
    x_lo = max(0, px - half)
    x_hi = min(W, px + half + 1)
    ix_win = Ix[y_lo:y_hi, x_lo:x_hi].ravel()
    iy_win = Iy[y_lo:y_hi, x_lo:x_hi].ravel()
    it_win = It[y_lo:y_hi, x_lo:x_hi].ravel()

    # 2. Build A·d = b  (brightness constancy: Ix·u + Iy·v + It = 0)
    A = np.column_stack([ix_win, iy_win])
    b = -it_win

    # 3. Normal equations: (AᵀA)d = Aᵀb
    AtA = A.T @ A
    Atb = A.T @ b

    # 4. Reject ill-conditioned windows (aperture problem)
    eigvals = np.linalg.eigvalsh(AtA)
    if eigvals.min() < 1e-6:
        return 0.0, 0.0

    # 5. Solve for flow d = [u, v]
    d = np.linalg.solve(AtA, Atb)
    return float(d[0]), float(d[1])


# --- Test your implementation ---
np.random.seed(42)
test_img1, test_img2, test_dx, test_dy = make_translating_blobs(dx=2.5, dy=1.5)
tIx, tIy, tIt = compute_gradients(test_img1, test_img2)

u_test, v_test = exercise_lk_single_point(tIx, tIy, tIt, 120, 90)
print(f"Your result:   u={u_test:.3f}, v={v_test:.3f}")
print(f"Ground truth:  u={test_dx:.3f}, v={test_dy:.3f}")
if abs(u_test) < 1e-8 and abs(v_test) < 1e-8:
    print("→ Flow is zero — check the implementation above.")
else:
    err = np.sqrt((u_test - test_dx)**2 + (v_test - test_dy)**2)
    print(f"→ Endpoint error: {err:.4f} px {'✓ Good!' if err < 0.5 else '— try again'}")

---

### Exercise 2: Track 100 Feature Points Across 30 Frames

Use OpenCV's pyramidal LK (`cv2.calcOpticalFlowPyrLK`) to track Shi-Tomasi corners
across a synthetic sequence. Plot the tracks and count how many survive.

In [ ]:
np.random.seed(42)
base_pattern = make_rotating_pattern(H=250, W=350, n_blobs=20)
ex2_frames = generate_rotating_sequence(base_pattern, n_frames=30,
                                        angle_step=1.0, tx=0.8, ty=0.3)

# 1. Detect Shi-Tomasi corners in first frame
frame0_u8 = np.clip(ex2_frames[0] * 255, 0, 255).astype(np.uint8)
corners = cv2.goodFeaturesToTrack(
    frame0_u8, maxCorners=100, qualityLevel=0.01, minDistance=8, blockSize=7
)
n_initial = len(corners)
n_frames = len(ex2_frames)

# 2. Track through all frames with pyramidal LK
lk_params = dict(
    winSize=(21, 21), maxLevel=3,
    criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01),
)
tracks = np.full((n_frames, n_initial, 2), np.nan)
tracks[0] = corners.reshape(-1, 2)
active = np.ones(n_initial, dtype=bool)
curr_pts = corners.copy()

for f in range(1, n_frames):
    prev_u8 = np.clip(ex2_frames[f - 1] * 255, 0, 255).astype(np.uint8)
    curr_u8 = np.clip(ex2_frames[f] * 255, 0, 255).astype(np.uint8)
    next_pts, status, _ = cv2.calcOpticalFlowPyrLK(prev_u8, curr_u8, curr_pts, None, **lk_params)
    status = status.ravel().astype(bool)
    active &= status
    curr_pts[~status] = -1
    tracks[f, active] = next_pts[active].reshape(-1, 2)
    curr_pts = next_pts

# 3. Plot tracks on last frame
fig, ax = plt.subplots(figsize=(12, 8))
ax.imshow(ex2_frames[-1], cmap="gray")
colors = plt.cm.tab10(np.linspace(0, 1, n_initial))
for j in range(n_initial):
    xs = tracks[:, j, 0]
    ys = tracks[:, j, 1]
    valid = ~np.isnan(xs)
    if valid.sum() > 1:
        ax.plot(xs[valid], ys[valid], "-", color=colors[j % 10], linewidth=1.2, alpha=0.8)
        ax.plot(xs[valid][-1], ys[valid][-1], "o", color=colors[j % 10], markersize=4)
ax.set_title("Pyramidal LK Tracks (30 frames)", fontsize=13)
ax.set_xlabel("x (px)"); ax.set_ylabel("y (px)")
plt.tight_layout()
plt.show()

# 4. Count survivors
survived = int(active.sum())
print(f"{survived} / {n_initial} points survived to frame 30")
assert survived > n_initial * 0.5, f"Expected >50% survival, got {survived}/{n_initial}"

---

### Exercise 3: Dense Optical Flow Visualization

Compute Farnebäck dense optical flow on a synthetic multi-object scene
and visualize it using the HSV color-wheel encoding.

In [ ]:
np.random.seed(123)

def make_exercise_scene(H=250, W=350):
    """Create a synthetic scene with three moving objects on textured background."""
    bg = np.random.rand(H, W) * 0.15 + 0.1
    bg = gaussian_filter(bg, sigma=3)

    img1 = bg.copy()
    img2 = bg.copy()

    rects = [
        {"y": 30, "x": 40, "h": 70, "w": 90, "val": 0.8, "dx": 6, "dy": 0},
        {"y": 120, "x": 180, "h": 50, "w": 80, "val": 0.6, "dx": 0, "dy": 5},
        {"y": 160, "x": 50, "h": 60, "w": 60, "val": 0.5, "dx": -4, "dy": -3},
    ]
    for r in rects:
        tex = r["val"] + np.random.randn(r["h"], r["w"]) * 0.06
        y, x, h, w = r["y"], r["x"], r["h"], r["w"]
        img1[y:y+h, x:x+w] = tex
        ny, nx = y + r["dy"], x + r["dx"]
        ny = np.clip(ny, 0, H - h)
        nx = np.clip(nx, 0, W - w)
        img2[ny:ny+h, nx:nx+w] = tex

    return np.clip(img1, 0, 1), np.clip(img2, 0, 1)

ex3_img1, ex3_img2 = make_exercise_scene()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].imshow(ex3_img1, cmap="gray")
axes[0].set_title("Frame 1")
axes[1].imshow(ex3_img2, cmap="gray")
axes[1].set_title("Frame 2")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

# 1. Convert to uint8
ex3_u8_1 = np.clip(ex3_img1 * 255, 0, 255).astype(np.uint8)
ex3_u8_2 = np.clip(ex3_img2 * 255, 0, 255).astype(np.uint8)

# 2. Farnebäck dense optical flow
flow = cv2.calcOpticalFlowFarneback(
    ex3_u8_1, ex3_u8_2, None,
    pyr_scale=0.5, levels=3, winsize=15,
    iterations=3, poly_n=5, poly_sigma=1.2, flags=0,
)
flow_mag = np.sqrt(flow[..., 0]**2 + flow[..., 1]**2)
print(f"Mean flow magnitude: {flow_mag.mean():.2f} px, max: {flow_mag.max():.2f} px")

# 3. Visualize: HSV color wheel + quiver overlay
flow_rgb = flow_to_color(flow)
fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(flow_rgb)
axes[0].set_title("Farnebäck flow (HSV color wheel)")
axes[0].axis("off")

step = 12
yy, xx = np.mgrid[0:flow.shape[0]:step, 0:flow.shape[1]:step]
axes[1].imshow(ex3_img1, cmap="gray")
axes[1].quiver(xx, yy, flow[::step, ::step, 0], flow[::step, ::step, 1],
               color="lime", angles="xy", scale_units="xy", scale=1, width=0.003)
axes[1].set_title("Flow vectors overlaid on frame 1")
axes[1].axis("off")

# Color wheel legend
wheel = np.zeros((128, 128, 3))
for yi in range(128):
    for xi in range(128):
        ang = np.arctan2(yi - 64, xi - 64)
        mag = np.sqrt((xi - 64)**2 + (yi - 64)**2) / 64.0
        wheel[yi, xi] = hsv_to_rgb(np.array([[(ang + np.pi) / (2 * np.pi), 1.0, min(mag, 1.0)]]))[0, 0]
axes[2].imshow(wheel)
axes[2].set_title("Color wheel: hue=direction, sat=magnitude")
axes[2].axis("off")
plt.tight_layout()
plt.show()

assert flow_mag.max() > 1.0, "Expected visible motion in synthetic scene"
print("Exercise 3 complete: dense flow computed and visualized.")

---

### Exercise 4: Scene Flow from Depth + Optical Flow

Given synthetic depth maps and 2D optical flow, compute and visualize 3D scene flow.
This exercise puts together everything from Section 7.

In [ ]:
np.random.seed(99)

ex4_H, ex4_W = 80, 120
ex4_fx, ex4_fy = 150.0, 150.0
ex4_cx, ex4_cy = ex4_W / 2, ex4_H / 2

ex4_K = np.array([[ex4_fx, 0, ex4_cx],
                   [0, ex4_fy, ex4_cy],
                   [0,  0,  1]], dtype=np.float64)
ex4_K_inv = np.linalg.inv(ex4_K)

yy4, xx4 = np.mgrid[0:ex4_H, 0:ex4_W].astype(np.float64)

ex4_depth1 = 4.0 + 0.8 * np.sin(xx4 * 0.06) * np.cos(yy4 * 0.08)

ex4_flow = np.zeros((ex4_H, ex4_W, 2))

obj_mask = ((xx4 - 60)**2 + (yy4 - 40)**2) < 20**2
ex4_flow[obj_mask, 0] = 3.0
ex4_flow[obj_mask, 1] = -2.0
ex4_depth1[obj_mask] = 2.5

ex4_depth2 = ex4_depth1.copy()
ex4_depth2[obj_mask] -= 0.4  # object approaches camera

print(f"Exercise 4 setup: {ex4_W}×{ex4_H} image, circular moving object")
print(f"Object 2D flow: (3, -2) px, depth change: -0.4 m")

# 1. Compute 3D scene flow
ex4_sf, ex4_P1, ex4_P2 = compute_scene_flow(
    ex4_depth1, ex4_depth2, ex4_flow, ex4_K_inv
)
ex4_sf_mag = np.sqrt(np.sum(ex4_sf**2, axis=-1))
ex4_flow_mag = np.sqrt(ex4_flow[..., 0]**2 + ex4_flow[..., 1]**2)

# 2. Mean 3D motion on the moving object
obj_sf = ex4_sf[obj_mask]
print(f"Mean scene flow on object: dX={obj_sf[:, 0].mean():.4f}, "
      f"dY={obj_sf[:, 1].mean():.4f}, dZ={obj_sf[:, 2].mean():.4f} m")
assert obj_sf[:, 2].mean() < -0.1, "Object approaches camera → negative dZ expected"
assert ex4_sf_mag[~obj_mask].mean() < 0.05, "Background should have near-zero 3D flow"

# 3. 3D quiver plot
fig = plt.figure(figsize=(12, 8))
ax = fig.add_subplot(111, projection="3d")
step = 3
xs = ex4_P1[::step, ::step, 0].ravel()
ys = ex4_P1[::step, ::step, 1].ravel()
zs = ex4_P1[::step, ::step, 2].ravel()
us = ex4_sf[::step, ::step, 0].ravel()
vs = ex4_sf[::step, ::step, 1].ravel()
ws = ex4_sf[::step, ::step, 2].ravel()
mags = np.sqrt(us**2 + vs**2 + ws**2)
moving = mags > 0.02
ax.scatter(xs[~moving], ys[~moving], zs[~moving], c="lightgray", s=2, alpha=0.4, label="Static")
if moving.any():
    q = ax.quiver(xs[moving], ys[moving], zs[moving],
                  us[moving], vs[moving], ws[moving],
                  length=1.5, normalize=False, arrow_length_ratio=0.3, linewidth=1.2)
    q.set_array(mags[moving])
    q.set_cmap("plasma")
    fig.colorbar(q, ax=ax, shrink=0.6, label="3D flow magnitude (m)")
ax.set_xlabel("X (m)"); ax.set_ylabel("Y (m)"); ax.set_zlabel("Z (m)")
ax.set_title("Exercise 4: 3D Scene Flow of Moving Object", fontsize=13)
ax.view_init(elev=25, azim=-55)
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

# 4. 2×2 diagnostic panel
fig, axes = plt.subplots(2, 2, figsize=(12, 9))
im0 = axes[0, 0].imshow(ex4_depth1, cmap="viridis")
axes[0, 0].set_title("Depth map (m)"); fig.colorbar(im0, ax=axes[0, 0], fraction=0.046)
im1 = axes[0, 1].imshow(ex4_flow_mag, cmap="hot")
axes[0, 1].set_title("2D flow magnitude (px)"); fig.colorbar(im1, ax=axes[0, 1], fraction=0.046)
im2 = axes[1, 0].imshow(ex4_sf_mag, cmap="plasma")
axes[1, 0].set_title("3D scene flow magnitude (m)"); fig.colorbar(im2, ax=axes[1, 0], fraction=0.046)
im3 = axes[1, 1].imshow(ex4_sf[..., 2], cmap="RdBu_r")
axes[1, 1].set_title("Z-component of scene flow (m)"); fig.colorbar(im3, ax=axes[1, 1], fraction=0.046)
for ax in axes.ravel():
    ax.set_xlabel("u (px)"); ax.set_ylabel("v (px)")
plt.suptitle("Scene Flow Decomposition", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()
print("Exercise 4 complete: 3D scene flow verified on synthetic moving object.")

---

### Exercise 5: RAFT vs Classical Flow Comparison

If GPU is available, compare RAFT against Farneback on the same image pair.
Otherwise, discuss the expected differences based on the architecture.

In [ ]:
# Exercise 5: Classical flow comparison with endpoint error (EPE)
# Compare single-scale LK (sparse) vs Farnebäck (dense) on known ground truth.

np.random.seed(7)
cmp_img1, cmp_img2, gt_u, gt_v = make_translating_blobs(dx=4.0, dy=-2.5)
cmp_Ix, cmp_Iy, cmp_It = compute_gradients(cmp_img1, cmp_img2)

# Sparse LK at Harris corners
cmp_u8_1 = np.clip(cmp_img1 * 255, 0, 255).astype(np.uint8)
corners_cmp = cv2.goodFeaturesToTrack(cmp_u8_1, maxCorners=50, qualityLevel=0.01, minDistance=10)
lk_epe = []
for pt in corners_cmp:
    px, py = int(pt[0, 0]), int(pt[0, 1])
    u_lk, v_lk = exercise_lk_single_point(cmp_Ix, cmp_Iy, cmp_It, px, py)
    lk_epe.append(np.sqrt((u_lk - gt_u)**2 + (v_lk - gt_v)**2))

# Dense Farnebäck
cmp_u8_2 = np.clip(cmp_img2 * 255, 0, 255).astype(np.uint8)
flow_fb = cv2.calcOpticalFlowFarneback(cmp_u8_1, cmp_u8_2, None, 0.5, 3, 15, 3, 5, 1.2, 0)
fb_epe_map = np.sqrt((flow_fb[..., 0] - gt_u)**2 + (flow_fb[..., 1] - gt_v)**2)

print("=== Classical Flow Comparison (ground truth: uniform translation) ===")
print(f"Ground truth flow: u={gt_u:.1f}, v={gt_v:.1f} px")
print(f"LK (sparse, {len(lk_epe)} points):  mean EPE = {np.mean(lk_epe):.3f} px")
print(f"Farnebäck (dense):                   mean EPE = {fb_epe_map.mean():.3f} px")

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].imshow(cmp_img1, cmap="gray")
axes[0].set_title("Input frame"); axes[0].axis("off")
im1 = axes[1].imshow(fb_epe_map, cmap="hot", vmin=0, vmax=2)
axes[1].set_title(f"Farnebäck EPE (mean={fb_epe_map.mean():.2f})")
fig.colorbar(im1, ax=axes[1], fraction=0.046)
axes[2].hist(lk_epe, bins=15, alpha=0.7, label=f"LK (mean={np.mean(lk_epe):.2f})")
axes[2].axvline(fb_epe_map.mean(), color="red", linestyle="--", label=f"Farnebäck mean={fb_epe_map.mean():.2f}")
axes[2].set_xlabel("Endpoint error (px)"); axes[2].set_ylabel("Count")
axes[2].set_title("EPE distribution"); axes[2].legend(fontsize=9)
plt.tight_layout()
plt.show()

# Single-scale LK can reach ~1 px on textured blobs; pyramidal LK is tighter.
lk_params_cmp = dict(winSize=(21, 21), maxLevel=3,
                     criteria=(cv2.TERM_CRITERIA_EPS | cv2.TERM_CRITERIA_COUNT, 30, 0.01))
next_pts, status, _ = cv2.calcOpticalFlowPyrLK(
    cmp_u8_1, cmp_u8_2, corners_cmp, None, **lk_params_cmp)
pyr_epe = []
for i in range(len(corners_cmp)):
    if status[i, 0]:
        u_p = next_pts[i, 0, 0] - corners_cmp[i, 0, 0]
        v_p = next_pts[i, 0, 1] - corners_cmp[i, 0, 1]
        pyr_epe.append(np.sqrt((u_p - gt_u)**2 + (v_p - gt_v)**2))
print(f"Pyramidal LK ({len(pyr_epe)} pts): mean EPE = {np.mean(pyr_epe):.3f} px")

assert np.mean(pyr_epe) < 0.5, f"Pyramidal LK EPE too high: {np.mean(pyr_epe):.2f}"
# Farnebäck averages over the whole image including zero-flow background;
# check that motion direction is correct in the high-gradient region.
mask_motion = cmp_Ix**2 + cmp_Iy**2 > np.percentile(cmp_Ix**2 + cmp_Iy**2, 75)
fb_u = flow_fb[..., 0][mask_motion].mean()
fb_v = flow_fb[..., 1][mask_motion].mean()
assert np.sign(fb_u) == np.sign(gt_u), f"Farnebäck u direction wrong: {fb_u:.2f} vs gt {gt_u:.1f}"
assert np.sign(fb_v) == np.sign(gt_v), f"Farnebäck v direction wrong: {fb_v:.2f} vs gt {gt_v:.1f}"
print(f"Farnebäck motion region: mean flow = ({fb_u:.2f}, {fb_v:.2f}) px — direction matches GT ✓")
print("RAFT (if GPU available) would typically achieve EPE < 0.5 px on Sintel/KITTI")
print("but requires PyTorch + pretrained weights — classical methods suffice for this workshop.")

---

## Summary

| Method | Type | Displacements | Speed | Accuracy |
|--------|------|---------------|-------|----------|
| **Lucas-Kanade** | Sparse | Small (< 1 px) | Fast | Good for corners |
| **Pyramidal LK** | Sparse | Medium (< 20 px) | Fast | Good |
| **Farnebäck** | Dense | Medium | Medium | Moderate |
| **RAFT** | Dense | Large | Slow (GPU) | State-of-the-art |
| **MegaFlow** | Dense | Very large | Slow (GPU) | SOTA (Sintel + KITTI) |
| **TAPIR / CoTracker** | Point tracks | Any | Slow (GPU) | Excellent |
| **Scene Flow** | 3D dense | Depends on 2D method | Varies | Depth-dependent |

### Key Equations

- **Optical flow constraint**: $I_x u + I_y v + I_t = 0$
- **Normal flow**: $v_n = -I_t / |\nabla I|$
- **Lucas-Kanade**: $(A^\top A) \mathbf{d} = A^\top \mathbf{b}$ where $A^\top A$ is the structure tensor
- **Pyramidal propagation**: $\mathbf{d}_{\text{init}}^\ell = 2 \, \mathbf{d}^{\ell+1}$
- **Scene flow**: $\Delta \mathbf{P} = Z_2 K^{-1} \tilde{\mathbf{p}}_2 - Z_1 K^{-1} \tilde{\mathbf{p}}_1$
- **RAFT loss**: $L = \sum_{i=1}^{N} \gamma^{N-i} \| \mathbf{f}_i - \mathbf{f}_{gt} \|_1$

### What's Next

In **Notebook 07** we'll use multi-view correspondences (which optical flow helps
establish) to recover camera motion via **visual odometry**.

---

## 8. Foundation Models for Dense Correspondence (2025–2026)

### The Paradigm Shift

For over a decade, optical flow research followed a clear trajectory: design task-specific
architectures with increasingly sophisticated inductive biases — correlation volumes (FlowNet),
iterative refinement (RAFT), all-pairs attention (GMA). Each generation improved accuracy but
remained a **single-purpose model** trained exclusively on flow data.

Starting in 2025, a new paradigm emerged: **foundation-model-driven flow inference**. Instead
of building flow-specific feature extractors from scratch, these methods leverage the rich
visual representations already learned by large-scale vision foundation models (DINOv2,
Depth Anything, SAM) and adapt them for correspondence estimation. The result is simpler
architectures that match or exceed task-specific models.

### WAFT — Do We Need Cost Volumes? (Wang et al., ICLR 2026 Oral)

**WAFT** (Warping-Alone Field Transforms) challenges the central assumption of RAFT: that
an all-pairs cost volume is necessary for strong performance. WAFT replaces the cost volume
with **high-resolution warping** — warp frame 2's features toward frame 1 using the current
flow estimate, then compute the residual directly.

**Key insight:** At 1/8 resolution, the all-pairs cost volume requires $O(H^2 W^2 / 64^2)$
memory. WAFT's warping-only design uses $O(HW/64)$ — a dramatic reduction that enables
higher-resolution processing.

| Method | Sintel Clean | Sintel Final | KITTI Fl-all | Spring 1px |
|--------|:-----------:|:------------:|:------------:|:----------:|
| RAFT | 1.60 | 2.85 | 5.10 | — |
| FlowFormer++ | 1.07 | 1.94 | 4.52 | — |
| DPFlow (CVPR 2025) | 1.04 | 1.97 | 3.56 | — |
| **WAFT-1080p** | **1.03** | **1.93** | **3.35** | **1.222** |

WAFT ranks **1st on Spring and KITTI**, with the best zero-shot generalization on KITTI,
while being up to **4.1× faster** than methods with similar performance.

### Optical Flow Matching — Flow as Transport Dynamics (Luo et al., CVPR 2026)

**OFM** (Optical Flow Matching) reframes optical flow as a **continuous-time transport**
problem rather than a discrete displacement prediction:

$$\frac{d\mathbf{x}}{dt} = \mathbf{v}_\theta(\mathbf{x}, t), \quad \mathbf{x}(0) = \text{pixel in frame 1}, \quad \mathbf{x}(1) = \text{correspondence in frame 2}$$

The network learns a velocity field $\mathbf{v}_\theta$ and integrates it via an Euler ODE
solver. This explicit velocity–displacement coupling produces **temporally coherent and
physically consistent flow** — a property critical for video-based autonomous navigation.

### DPFlow — Adaptive Multi-Resolution (Morimitsu et al., CVPR 2025)

**DPFlow** addresses a practical limitation: most flow networks train at fixed resolution
(~448×1024) and degrade at higher resolutions. DPFlow uses a **dual-pyramid** encoder
that brings information from deep (semantic) features to shallow (fine-grained) features,
enabling a single model to handle inputs from 448p to **8K** resolution. At 8K, DPFlow
outperforms RAPIDFlow by 30%.

### FlowSeek — Depth Foundation Models for Flow (Poggi et al., ICCV 2025)

**FlowSeek** demonstrates that **foundation depth priors** can replace much of the learned
flow architecture. The key insight: depth maps decompose the scene into geometric layers,
and within each layer the flow is locally rigid (ego-motion) or smoothly varying:

1. Extract **monocular depth** via a foundation model (Depth Anything v2)
2. Decompose the flow field into **motion bases** (one per depth layer)
3. Estimate per-pixel coefficients for the motion bases — a much simpler regression problem

**Result:** FlowSeek achieves competitive accuracy with **2.81 EPE on Sintel Final** versus
RAFT's 2.86, while requiring no iterative refinement at test time.

### Implications for Autonomous Systems

These developments have direct consequences for the VIO/SLAM pipelines covered in later
notebooks:

- **Calibrated uncertainty**: SEA-RAFT's Mixture-of-Laplacians (MoL) loss produces
  per-pixel flow distributions, not just point estimates. These uncertainty estimates can
  be propagated into visual-inertial odometry and SLAM as **measurement covariances**,
  replacing hand-tuned noise parameters with data-driven confidence.
- **Domain generalization**: Foundation model features transfer across domains (indoor,
  outdoor, aerial, underwater) without fine-tuning, reducing the engineering burden of
  deploying flow in new environments.
- **Unified perception backbone**: A single foundation model can serve optical flow, depth
  estimation, and semantic segmentation simultaneously, reducing compute and memory on
  resource-constrained platforms.

> **Takeaway:** The field is moving from "design a better flow architecture" to "adapt a
> foundation model for correspondence." This mirrors the broader trend in computer vision
> where task-specific models are being replaced by foundation-model adaptations.